# IT3051 - Fundamentals of Data Mining & Machine Learning
## Mini Project 2026: Used Car Price Prediction System
### Stage 6: Model Development — Linear Regression (OLS Baseline)

**Assigned Model:** Ordinary Least Squares (OLS) Linear Regression  
**Target Variable:** `log_price` (log-transformed vehicle selling price)  
**Input Features:** 13 engineered and scaled features from `Used_Car_Price_Prediction_System.ipynb`

In [ ]:
# 1. Import Core Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold, cross_validate
from sklearn.metrics import r2_score, mean_absolute_error, root_mean_squared_error

pd.set_option('display.float_format', '{:,.4f}'.format)

---
## 2. Load Preprocessed Datasets
We load the standardized training and testing datasets created by `Used_Car_Price_Prediction_System.ipynb`.

In [ ]:
# Load preprocessed datasets
X_train = pd.read_csv('X_train.csv')
X_test  = pd.read_csv('X_test.csv')
y_train = pd.read_csv('y_train.csv').values.ravel()
y_test  = pd.read_csv('y_test.csv').values.ravel()

print(f"X_train shape: {X_train.shape} (3,206 samples, 13 features)")
print(f"X_test  shape: {X_test.shape} (802 samples, 13 features)")
print(f"Features list: {list(X_train.columns)}")

---
## 3. Baseline Training & 5-Fold Cross-Validation
We train an Ordinary Least Squares (OLS) Linear Regression model using **5-Fold Cross-Validation** strictly on the training set (`random_state=42`).

In [ ]:
# Define baseline model and 5-fold CV
kf = KFold(n_splits=5, shuffle=True, random_state=42)
lr_model = LinearRegression()

# Perform cross-validation on training data only
cv_results = cross_validate(
    lr_model,
    X_train,
    y_train,
    cv=kf,
    scoring={
        'r2': 'r2',
        'mae': 'neg_mean_absolute_error',
        'rmse': 'neg_root_mean_squared_error'
    },
    n_jobs=-1
)

cv_r2_mean   = cv_results['test_r2'].mean()
cv_r2_std    = cv_results['test_r2'].std()
cv_mae_mean  = -cv_results['test_mae'].mean()
cv_mae_std   = cv_results['test_mae'].std()
cv_rmse_mean = -cv_results['test_rmse'].mean()
cv_rmse_std  = cv_results['test_rmse'].std()

print("=== 5-Fold Cross-Validation Results (Training Set) ===")
print(f"Mean CV R²   : {cv_r2_mean:.4f} (+/- {cv_r2_std:.4f})")
print(f"Mean CV MAE  : {cv_mae_mean:.4f} (+/- {cv_mae_std:.4f})")
print(f"Mean CV RMSE : {cv_rmse_mean:.4f} (+/- {cv_rmse_std:.4f})")

---
## 4. Evaluation on Untouched Test Set
We fit the model on the full training set and evaluate its generalization on `X_test`. Both log-scale metrics and inverse-transformed dollar metrics (`expm1`) are reported.

In [ ]:
# Fit on full training data
lr_model.fit(X_train, y_train)

# Predict on untouched test data
y_pred = lr_model.predict(X_test)

test_r2   = r2_score(y_test, y_pred)
test_mae  = mean_absolute_error(y_test, y_pred)
test_rmse = root_mean_squared_error(y_test, y_pred)

# Dollar scale via expm1
y_test_dollar = np.expm1(y_test)
y_pred_dollar = np.expm1(y_pred)
test_mae_dollar  = mean_absolute_error(y_test_dollar, y_pred_dollar)
test_rmse_dollar = root_mean_squared_error(y_test_dollar, y_pred_dollar)

print("=== Linear Regression Test Set Performance ===")
print(f"Test R² (log scale)   : {test_r2:.4f}")
print(f"Test MAE (log scale)  : {test_mae:.4f}")
print(f"Test RMSE (log scale) : {test_rmse:.4f}")
print("-" * 50)
print(f"Test MAE ($ scale)    : ${test_mae_dollar:,.2f}")
print(f"Test RMSE ($ scale)   : ${test_rmse_dollar:,.2f}")

---
## 5. Model Coefficients & Interpretability
Linear Regression coefficients reveal the estimated linear effect of each feature on `log_price`.

In [ ]:
# Coefficients inspection
coef_df = pd.DataFrame({
    'Feature': X_train.columns,
    'Coefficient': lr_model.coef_
}).sort_values('Coefficient', key=abs, ascending=False)

print("=== Linear Regression Feature Coefficients ===")
print(coef_df.to_string(index=False))

---
## 6. Save Model Evaluation Record

In [ ]:
# Save model results record
import os
os.makedirs('results', exist_ok=True)

results_record = {
    'Model': 'Linear Regression',
    'Version': 'Baseline',
    'Parameters': 'default (fit_intercept=True)',
    'CV_R2': cv_r2_mean,
    'CV_R2_Std': cv_r2_std,
    'CV_MAE': cv_mae_mean,
    'CV_RMSE': cv_rmse_mean,
    'Test_R2': test_r2,
    'Test_MAE': test_mae,
    'Test_RMSE': test_rmse,
    'Test_MAE_Dollar': test_mae_dollar,
    'Test_RMSE_Dollar': test_rmse_dollar,
    'Observation': 'Simple linear parametric baseline. Fast and interpretable; bounded by linear assumption.'
}

pd.DataFrame([results_record]).to_csv(os.path.join('results', 'linear_regression_results.csv'), index=False)
print("Linear Regression evaluation saved to results/linear_regression_results.csv")